# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
print(spaceship.isnull().sum())

spaceship.shape

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


(8693, 14)

In [4]:
spaceship_clean = spaceship.dropna()
spaceship_clean.shape


(6606, 14)

In [5]:
spaceship_clean = spaceship_clean.drop(columns=['PassengerId', 'Name'])

non_numerical_columns = spaceship_clean.select_dtypes(include=['object', 'bool']).columns.tolist()

# Remove target from the list
if 'Transported' in non_numerical_columns:
    non_numerical_columns.remove('Transported')



print(non_numerical_columns)

['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']


In [7]:
spaceship_dummies = pd.get_dummies(spaceship_clean, columns=non_numerical_columns, drop_first=False)

print(spaceship_dummies.shape)
print(spaceship_dummies.columns)

(6606, 5322)
Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Transported', 'HomePlanet_Earth', 'HomePlanet_Europa',
       'HomePlanet_Mars',
       ...
       'Cabin_G/998/S', 'Cabin_G/999/P', 'Cabin_G/999/S', 'Cabin_T/1/P',
       'Cabin_T/3/P', 'Destination_55 Cancri e', 'Destination_PSO J318.5-22',
       'Destination_TRAPPIST-1e', 'VIP_False', 'VIP_True'],
      dtype='object', length=5322)


**Perform Train Test Split**

In [8]:

X = spaceship_dummies.drop(columns=['Transported'])
y = spaceship_dummies['Transported']

# Split into train and test sets (80-20 %)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
       #42

print(f"Training set: {X_train.shape[0]} ")
print(f"Test set: {X_test.shape[0]} ")

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Training set: 5284 
Test set: 1322 
Features shape: (6606, 5321)
Target shape: (6606,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [12]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

- Bagging and Pasting

In [13]:



bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, max_samples=0.8, random_state=42, n_jobs=-1)

bagging_model.fit(X_train, y_train)
y_pred_bagging = bagging_model.predict(X_test)

bagging_accuracy = accuracy_score(y_test, y_pred_bagging)
print(f"\nBagging Accuracy: {bagging_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bagging))


Bagging Accuracy: 0.8079

Classification Report:
              precision    recall  f1-score   support

       False       0.82      0.79      0.80       653
        True       0.80      0.83      0.81       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



In [18]:
pasting_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, max_samples=0.8, bootstrap=False, random_state=42, n_jobs=-1)

pasting_model.fit(X_train, y_train)
y_pred_pasting = pasting_model.predict(X_test)

pasting_accuracy = accuracy_score(y_test, y_pred_pasting)
print(f"\nPasting Accuracy: {pasting_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_pasting))


Pasting Accuracy: 0.7980

Classification Report:
              precision    recall  f1-score   support

       False       0.81      0.77      0.79       653
        True       0.79      0.83      0.81       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



- Random Forests

In [14]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=2, max_features='sqrt', random_state=42,n_jobs=-1)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f"\nRandom Forest Accuracy: {rf_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.7413

Classification Report:
              precision    recall  f1-score   support

       False       0.70      0.83      0.76       653
        True       0.80      0.66      0.72       669

    accuracy                           0.74      1322
   macro avg       0.75      0.74      0.74      1322
weighted avg       0.75      0.74      0.74      1322



- Gradient Boosting

In [15]:
gb_model = GradientBoostingClassifier( n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=5, min_samples_leaf=2, subsample=0.8, random_state=42)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

gb_accuracy = accuracy_score(y_test, y_pred_gb)
print(f"\nGradient Boosting Accuracy: {gb_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))


Gradient Boosting Accuracy: 0.7973

Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.75      0.78       653
        True       0.77      0.85      0.81       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



- Adaptive Boosting

In [16]:
ada_model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),n_estimators=100,learning_rate=1.0,random_state=42)

ada_model.fit(X_train, y_train)
y_pred_ada = ada_model.predict(X_test)

ada_accuracy = accuracy_score(y_test, y_pred_ada)
print(f"\nAdaptive Boosting Accuracy: {ada_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ada))


Adaptive Boosting Accuracy: 0.7746

Classification Report:
              precision    recall  f1-score   support

       False       0.76      0.80      0.78       653
        True       0.79      0.75      0.77       669

    accuracy                           0.77      1322
   macro avg       0.78      0.77      0.77      1322
weighted avg       0.78      0.77      0.77      1322



Which model is the best and why?

In [26]:
results = pd.DataFrame({'Model': ['Bagging', 'Pasting', 'Random Forest', 'Gradient Boosting', 'AdaBoost'], 'Accuracy': [bagging_accuracy, pasting_accuracy, rf_accuracy, gb_accuracy, ada_accuracy]}).sort_values('Accuracy', ascending=False)

print(f"\nBest Model: {best_model_name} with {best_accuracy:.4f} accuracy")


Best Model: Bagging with 0.8079 accuracy
